# U8-03 · Regularización, regresión y eficiencia

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 8, apartados 8.4, 8.5, 8.7 y 8.9*

Ejecuta cada celda con **Mayús + Intro**. Solo se usan NumPy y scikit-learn; no hace falta tarjeta gráfica.

## 1. La tasa de aprendizaje (ejemplo 3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0)
esc = StandardScaler().fit(X_tr)
A, B = esc.transform(X_tr), esc.transform(X_te)

for nombre, opc in [("SGD 0,001", dict(solver="sgd", learning_rate_init=0.001)),
                    ("SGD 0,1", dict(solver="sgd", learning_rate_init=0.1)),
                    ("SGD 1", dict(solver="sgd", learning_rate_init=1.0)),
                    ("Adam", dict(solver="adam"))]:
    red = MLPClassifier(hidden_layer_sizes=(64,), max_iter=100,
                        random_state=0, **opc).fit(A, y_tr)
    plt.plot(red.loss_curve_, label=nombre)
plt.yscale("log"); plt.xlabel("época"); plt.ylabel("pérdida"); plt.legend(); plt.show()

## 2. Regularización con 30 puntos (ejemplo 4)

In [ ]:
import numpy as np
from sklearn.neural_network import MLPRegressor

rng = np.random.default_rng(0)
x = rng.uniform(0, 1, 30)                     # 30 de entrenamiento
y = np.sin(2 * np.pi * x) + rng.normal(0, 0.3, 30)
x_n = rng.uniform(0, 1, 1000)                 # 1000 nuevos
y_n = np.sin(2 * np.pi * x_n) + rng.normal(0, 0.3, 1000)

for alpha in [0, 0.001, 0.01, 0.1, 1, 10]:
    red = MLPRegressor(hidden_layer_sizes=(100, 100),
                       activation="tanh", solver="lbfgs", alpha=alpha,
                       max_iter=5000, random_state=0)
    red.fit(x[:, None], y)
    ecm = lambda a, b: np.mean((red.predict(a[:, None]) - b) ** 2)
    print(alpha, round(ecm(x, y), 3), round(ecm(x_n, y_n), 3))

In [ ]:
malla = np.linspace(0, 1, 300)
plt.scatter(x, y, color="k", s=12, label="datos")
for alpha in [0, 0.1, 10]:
    red = MLPRegressor(hidden_layer_sizes=(100, 100), activation="tanh",
                       solver="lbfgs", alpha=alpha, max_iter=5000,
                       random_state=0).fit(x[:, None], y)
    plt.plot(malla, red.predict(malla[:, None]), label=f"alpha={alpha}")
plt.ylim(-2.5, 2.5); plt.legend(); plt.show()

**Prueba.** Sube los datos de entrenamiento de 30 a 300 y repite sin regularizar (`alpha=0`). ¿Sigue sobreajustando tanto?

## 3. La progresión de la diabetes (ejemplo 6)

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor

X, y = load_diabetes(return_X_y=True)        # 442 pacientes, 10 var.
cv = KFold(5, shuffle=True, random_state=0)

def red(capas, alpha):
    mlp = MLPRegressor(hidden_layer_sizes=capas, alpha=alpha,
                       max_iter=3000, random_state=0)
    tubo = make_pipeline(StandardScaler(), mlp)
    return TransformedTargetRegressor(          # escala también y
        tubo, transformer=StandardScaler())

lineal = make_pipeline(StandardScaler(), LinearRegression())
modelos = {"lineal": lineal,
           "red (32,)": red((32,), 1.0),
           "red (64, 64)": red((64, 64), 1.0),
           "red (64, 64) sin regularizar": red((64, 64), 1e-4)}
for nombre, m in modelos.items():
    r2 = cross_val_score(m, X, y, cv=cv, scoring="r2")
    print(f"{nombre:30s} R2 = {r2.mean():.3f}")

## 4. Medir la eficiencia (ejemplo 8)

Los tiempos dependen del equipo; en el navegador serán mayores. Fíjate en las proporciones.

In [ ]:
import time
import numpy as np
from sklearn.neural_network import MLPClassifier

def medir(capas, tipo, **opciones):
    red = MLPClassifier(hidden_layer_sizes=capas, max_iter=500,
                        random_state=0, **opciones)
    t0 = time.perf_counter()
    red.fit(A.astype(tipo), y_tr)
    seg = time.perf_counter() - t0
    params = sum(w.size + b.size
                 for w, b in zip(red.coefs_, red.intercepts_))
    kb = sum(w.nbytes + b.nbytes
             for w, b in zip(red.coefs_, red.intercepts_)) / 1024
    acierto = red.score(B.astype(tipo), y_te)
    print(capas, tipo.__name__, opciones, params, f"{kb:.0f} KB",
          red.n_iter_, "épocas", f"{seg:.2f} s", round(acierto, 4))

medir((256, 256), np.float64)
medir((256, 256), np.float32)
medir((256, 256), np.float64, early_stopping=True)
medir((16,), np.float64)

**Ejercicio.** Añade `medir((32,), np.float32)`. ¿Qué acierto da con cuánta memoria?

> Goodfellow, I., Bengio, Y. y Courville, A. (2016). *Deep Learning*. MIT Press. ISBN 978-0-262-03561-3
>
> Micikevicius, P. et al. (2018). Mixed Precision Training. *ICLR*. https://doi.org/10.48550/arXiv.1710.03740